In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut, RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

df = pd.read_csv("components.csv")

X = df.drop(columns=["measurement","no","id"]).values
y = df["measurement"].values

model = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", Ridge(alpha=1.0))
])

print(X.shape)
print(X[:5])  # Print first 5 rows of X
print(y.shape)
print(y[:5]) # Print first 5 values of y

In [ ]:

# --- LOOCV ---
loo = LeaveOneOut()
preds_loo = []
truth_loo = []

for train_idx, test_idx in loo.split(X):
    model.fit(X[train_idx], y[train_idx])
    preds_loo.append(model.predict(X[test_idx])[0])
    truth_loo.append(y[test_idx][0])

rmse_loo = np.sqrt(mean_squared_error(truth_loo, preds_loo))
print("LOOCV RMSE:", rmse_loo)

# --- Repeated K-Fold ---
rkf = RepeatedKFold(n_splits=4, n_repeats=10, random_state=42)
preds_rkf = []
truth_rkf = []

for train_idx, test_idx in rkf.split(X):
    model.fit(X[train_idx], y[train_idx])
    preds_rkf.extend(model.predict(X[test_idx]))
    truth_rkf.extend(y[test_idx])

rmse_rkf = np.sqrt(mean_squared_error(truth_rkf, preds_rkf))
print("Repeated K-Fold RMSE:", rmse_rkf)


In [ ]:
coef = model.named_steps["reg"].coef_
for name, c in zip(df.columns[:-1].drop(["no","id"]), coef):
    print(name, c)

In [ ]:
df2 = pd.read_csv("predict.csv")

X2 = df2.drop(columns=["no","id"]).values

preds = model.predict(X2)    

print("Predicted inrush currents for new data:")
#for i, pred in enumerate(preds):
#    print(f"Sample {i+1}: {pred:.2f}")

# for easy copy-paste
for i, pred in enumerate(preds):
    print(f"{pred:.2f} ")



In [ ]:
x_new = np.array([
    new_transformer["pri"],
    new_transformer["sec"],
    new_transformer["va"],
    new_transformer["dcr"],
    new_transformer["size"],
    new_transformer["width"],
    new_transformer["bobbin_width"],
    new_transformer["bobbin_depth"],
    new_transformer["pri_dia"],
    new_transformer["pri_turns"],
    new_transformer["io"],
    new_transformer["il"],
    new_transformer["coil_height"],
    new_transformer["bmax"],
]).reshape(1, -1)

for i, col in enumerate(df.columns[:-1].drop(["no","id"])):
    v = float(x_new[0, i])
    lo, hi = float(df[col].min()), float(df[col].max())
    if not ((lo <= v) and (v <= hi)):
        print(f"Warning: {col}={v} is outside training range [{lo}, {hi}]")

predicted_inrush = model.predict(x_new)
print("Predicted inrush current:", predicted_inrush[0])


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.metrics import mean_squared_error

kernel = ConstantKernel(1.0, (1e-4, 1e4)) * \
         RBF(length_scale=1.0, length_scale_bounds=(1e-8, 1e4)) + \
         WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e1))
reg = GaussianProcessRegressor(kernel=kernel)


df = pd.read_csv("components.csv")

X = df.drop(columns=["measurement","no","id"]).values
y = df["measurement"].values


model = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", reg)
])


print(X.shape)
print(X[:5])  # Print first 5 rows of X
print(y.shape)
print(y[:5]) # Print first 5 values of y

In [ ]:
# --- LOOCV Evaluation ---
loo = LeaveOneOut()
preds = []
truth = []
stds = []

for train_idx, test_idx in loo.split(X):
    model.fit(X[train_idx], y[train_idx])
    mean, std = model.predict(X[test_idx], return_std=True)
    preds.append(mean[0])
    stds.append(std[0])
    truth.append(y[test_idx][0])

rmse = np.sqrt(mean_squared_error(truth, preds))
print("LOOCV RMSE:", rmse)

# Optional: print average uncertainty
print("Mean predictive std:", np.mean(stds))
